# 3: Hypothesis Testing

This notebook turns the strongest descriptive patterns from the EDA into formal statistical tests. I use chi square tests for categorical service features at the arrival level and one way ANOVA for weather-related comparisons at the hourly level. Temperature and wind speed are back in the formal test set, while the results table still stays concise and reports only the test statistic, degrees of freedom, p value, and decision.

## 3.1 Setup and output paths

This section loads the core libraries used in the formal tests, points the notebook to the final merged dataset, and prepares the output folder for the hypothesis tables.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display
from scipy.stats import chi2_contingency, f_oneway

NOTEBOOK_DIR = Path.cwd().resolve()
ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
DATASET_PATH = ROOT / "data" / "processed" / "dataset_final.csv"
OUTPUTS_DIR = ROOT / "outputs"
TABLES_DIR = OUTPUTS_DIR / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")
print(f"Dataset path: {DATASET_PATH.relative_to(ROOT)}")
print(f"Tables directory: {TABLES_DIR.relative_to(ROOT)}")

Project root: /Users/berenaydogan/Documents/4.2/DSA210/Project/DSA210-Project
Dataset path: data/processed/dataset_final.csv
Tables directory: outputs/tables


## 3.2 Load the processed dataset

This section reads the merged dataset and derives the month field used in the month-based hypothesis.

In [2]:
# Parse every timestamp needed for the calendar features and the hourly weather aggregation.
assert DATASET_PATH.exists(), "Run data_collection.ipynb first so data/processed/dataset_final.csv exists."

date_columns = ["service_date", "scheduled_arrival", "scheduled_arrival_date", "weather_time"]
df = pd.read_csv(DATASET_PATH, parse_dates=date_columns)

df["service_month"] = df["service_date"].dt.month

## 3.3 Build the samples at the arrival level and the hourly weather level

This section creates the two analysis tables used throughout the notebook. I keep arrival-level records for the service-related chi square tests, and I build one row per weather hour for the ANOVA comparisons on hourly delay rates.

In [3]:
arrival_sample_overview = pd.DataFrame(
    {
        "metric": [
            "rows at the arrival level",
            "late arrivals",
            "on-time-or-early arrivals",
            "delay rate",
            "unique train types",
            "unique lines",
            "unique operators",
            "service months used for H3",
        ],
        "value": [
            len(df),
            int(df["delayed"].sum()),
            int((1 - df["delayed"]).sum()),
            f"{df['delayed'].mean():.2%}",
            df["train_type"].nunique(),
            df["line_name"].nunique(),
            df["operator"].nunique(),
            ", ".join(map(str, sorted(df["service_month"].unique()))),
        ],
    }
)

weather_hourly = (
    # Collapse arrivals to one row per matched weather hour so each ANOVA group uses independent hourly summaries.
    df.groupby("weather_time", as_index=False)
    .agg(
        arrival_records=("delayed", "size"),
        delayed_count=("delayed", "sum"),
        temperature_2m=("temperature_2m", "first"),
        precipitation=("precipitation", "first"),
        snowfall=("snowfall", "first"),
        wind_speed_10m=("wind_speed_10m", "first"),
    )
    .sort_values("weather_time")
)
weather_hourly["delay_rate"] = weather_hourly["delayed_count"] / weather_hourly["arrival_records"]
weather_hourly["rainy_hour"] = (weather_hourly["precipitation"] > 0).astype(int)
weather_hourly["snowy_hour"] = (weather_hourly["snowfall"] > 0).astype(int)


def make_quantile_labels(series: pd.Series, prefix: str) -> pd.Series:
    quantile_codes = pd.qcut(series, q=4, labels=False, duplicates="drop")
    return quantile_codes.map(
        lambda value: f"{prefix} quartile {int(value) + 1}" if pd.notna(value) else pd.NA
    )


weather_hourly["temperature_group"] = make_quantile_labels(weather_hourly["temperature_2m"], "Temperature")
weather_hourly["wind_speed_group"] = make_quantile_labels(weather_hourly["wind_speed_10m"], "Wind speed")

weather_sample_overview = pd.DataFrame(
    {
        "metric": [
            "hourly weather observations",
            "hours with rain",
            "hours with snow",
            "mean arrivals per matched hour",
            "median arrivals per matched hour",
            "temperature groups used for H8",
            "wind speed groups used for H11",
        ],
        "value": [
            len(weather_hourly),
            int(weather_hourly["rainy_hour"].sum()),
            int(weather_hourly["snowy_hour"].sum()),
            round(weather_hourly["arrival_records"].mean(), 2),
            round(weather_hourly["arrival_records"].median(), 2),
            weather_hourly["temperature_group"].nunique(dropna=True),
            weather_hourly["wind_speed_group"].nunique(dropna=True),
        ],
    }
)

print(f"Arrival-level sample size (n_arrivals): {len(df):,}")
print(f"Weather-hour sample size (n_weather_hours): {len(weather_hourly):,}")
display(arrival_sample_overview)
display(weather_sample_overview)

Arrival-level sample size (n_arrivals): 90,818
Weather-hour sample size (n_weather_hours): 2,743


,metric,value
0,rows at the arrival level,90818
1,late arrivals,43058
2,on-time-or-early arrivals,47760
3,delay rate,47.41%
4,unique train types,9
5,unique lines,27
6,unique operators,2
7,service months used for H3,"1, 4, 7, 10"


,metric,value
0,hourly weather observations,2743.00
1,hours with rain,586.00
2,hours with snow,18.00
3,mean arrivals per matched hour,33.11
4,median arrivals per matched hour,35.00
5,temperature groups used for H8,4.00
6,wind speed groups used for H11,4.00
